# 🎨 Text-to-3D: Tối Ưu Hóa Mô Hình Sinh 3D Từ Văn Bản

Notebook này bao gồm:
1. **Baseline**: DreamFusion với Stable Diffusion / DeepFloyd IF
2. **Tối ưu 1**: Progressive Resolution Training (64→128→256)
3. **Tối ưu 2**: Anti-Janus (Enhanced Prompt + Perp-Neg)
4. **Tối ưu 3**: All-in-One (Progressive + Hybrid SDS/SDI + Adaptive Guidance + Anti-Janus)
5. **Export & Visualize**: Xuất mesh 3D và lưu kết quả

---
## 📦 1. Setup & Cài Đặt Dependencies

In [ ]:
!git clone https://github.com/Tiens0710/TextTo3D.git
%cd TextTo3D/

In [ ]:
from huggingface_hub import interpreter_login

interpreter_login()

In [ ]:
# Core dependencies
!pip install ninja
!pip install lightning==2.0.0 omegaconf==2.3.0 jaxtyping typeguard \
    diffusers==0.20.0 transformers==4.30.2 accelerate \
    opencv-python tensorboard matplotlib imageio imageio[ffmpeg] \
    trimesh bitsandbytes sentencepiece safetensors huggingface_hub \
    xatlas networkx pysdf PyMCubes wandb torchmetrics controlnet_aux

# Zero123 dependencies
!pip install einops kornia taming-transformers-rom1504 git+https://github.com/openai/CLIP.git

# Visualization
!pip install open3d plotly

# Fix version conflicts
!pip install libigl==2.5.1
!pip install huggingface_hub==0.25.*

In [ ]:
# Build CUDA dependencies (takes a few minutes)
!pip install git+https://github.com/ashawkey/envlight.git
!pip install git+https://github.com/KAIR-BAIR/nerfacc.git@v0.5.2
!pip install git+https://github.com/NVlabs/nvdiffrast.git
!pip install git+https://github.com/NVlabs/tiny-cuda-nn/#subdirectory=bindings/torch

In [ ]:
# Verify installation
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

# Verify threestudio modules
import threestudio
print(f"\nthreestudio version: {threestudio.__version__}")

# Verify optimized modules are registered
modules_to_check = [
    'stable-diffusion-adaptive-guidance',
    'stable-diffusion-hybrid-guidance',
    'hybrid-sds-sdi-system',
    'enhanced-sd-prompt-processor',
]
for name in modules_to_check:
    try:
        cls = threestudio.find(name)
        print(f"  ✅ {name} → {cls.__name__}")
    except KeyError:
        print(f"  ❌ {name} NOT FOUND")

---
## ✏️ 2. Chọn Prompt

Thay đổi `prompt` bên dưới để sinh mô hình 3D mong muốn.

In [ ]:
prompt = "a DSLR photo of a corgi taking a selfi"

# Các prompt mẫu khác:
# prompt = "a zoomed out DSLR photo of a baby panda sitting on top of a stack of pancakes"
# prompt = "a DSLR photo of a blue jay standing on a pile of gold coins"
# prompt = "a DSLR photo of a robot made of sushi"
# prompt = "a beautiful dress made out of garbage bags"

print(f"Prompt: {prompt}")

---
## 🔵 3. Baseline – DreamFusion

Chạy phương pháp gốc để có kết quả so sánh.

### 3a. Stable Diffusion (nhẹ, ~6GB VRAM)

In [ ]:
!python launch.py \
    --config configs/dreamfusion-sd.yaml \
    --train --gpu 0 \
    system.prompt_processor.prompt="$prompt" \
    system.prompt_processor.spawn=false \
    trainer.max_steps=10000

### 3b. DeepFloyd IF (chất lượng cao hơn, ~20GB VRAM)

In [ ]:
# Uncomment dòng dưới nếu dùng GPU ≥ 20GB VRAM (A100)
# !python launch.py \
#     --config configs/dreamfusion-if.yaml \
#     --train --gpu 0 \
#     system.prompt_processor.prompt="$prompt" \
#     system.prompt_processor.spawn=false \
#     trainer.max_steps=10000

---
## 🚀 4. Tối Ưu 1 – Progressive Resolution Training

Training bắt đầu ở resolution thấp (64×64, batch=4), tăng dần lên 128→256.

**Ưu điểm**: Nhanh hơn ~30-40% so với baseline, giữ nguyên chất lượng.

| Giai đoạn | Steps | Resolution | Batch Size |
|-----------|-------|------------|------------|
| 1 | 0–2000 | 64×64 | 4 |
| 2 | 2000–5000 | 128×128 | 2 |
| 3 | 5000–10000 | 256×256 | 1 |

In [ ]:
!python launch.py \
    --config configs/dreamfusion-sd-progressive.yaml \
    --train --gpu 0 \
    system.prompt_processor.prompt="$prompt" \
    system.prompt_processor.spawn=false \
    trainer.max_steps=10000

---
## 🛡️ 5. Tối Ưu 2 – Anti-Janus (Giảm Multi-Face)

Sử dụng Enhanced Prompt Processor + Perp-Neg để giảm hiện tượng Janus.

**Cải tiến**:
- Thêm negative prompt cho back view: `face, eyes, mouth, front view, looking at viewer`
- Thêm negative prompt cho side view: `front view, face`
- Scale back-view negative embedding ×2.0
- Bật Perpendicular Negative Prompting

In [ ]:
!python launch.py \
    --config configs/dreamfusion-sd-antijanus.yaml \
    --train --gpu 0 \
    system.prompt_processor.prompt="$prompt" \
    system.prompt_processor.spawn=false \
    trainer.max_steps=10000

---
## 🎯 6. Tối Ưu 3 – All-in-One (Tất Cả Cải Tiến)

Kết hợp tất cả 5 kỹ thuật tối ưu:

| Kỹ thuật | Mô tả |
|----------|--------|
| Progressive Resolution | 64→128→256, batch 4→2→1 |
| Hybrid SDS+SDI | SDS (step 0–3000) → Blend → SDI (step 4000+) |
| Adaptive Guidance Scale | 100 → 50 (giảm dần) |
| Enhanced Anti-Janus | Stronger back/side negative prompts |
| Timestep Annealing | max\_step\_percent giảm tuyến tính |

In [ ]:
!python launch.py \
    --config configs/optimized-dreamfusion.yaml \
    --train --gpu 0 \
    system.prompt_processor.prompt="$prompt" \
    system.prompt_processor.spawn=false \
    trainer.max_steps=10000

---
## 📤 7. Export Mesh 3D

Chọn thư mục output từ training run ở trên.

In [ ]:
import os
import glob

# Tìm output mới nhất
output_dirs = sorted(glob.glob('outputs/*/'), key=os.path.getmtime, reverse=True)

print("📁 Các output có sẵn:")
for i, d in enumerate(output_dirs[:10]):
    subdirs = sorted(glob.glob(os.path.join(d, '*/')), key=os.path.getmtime, reverse=True)
    for sd in subdirs[:3]:
        print(f"  [{i}] {sd}")

# Chọn output mới nhất
if output_dirs:
    latest = sorted(glob.glob(os.path.join(output_dirs[0], '*/')), key=os.path.getmtime, reverse=True)
    if latest:
        save_dir = latest[0].rstrip('/')
        print(f"\n✅ Đã chọn: {save_dir}")
    else:
        print("❌ Không tìm thấy output")
else:
    print("❌ Chưa có output. Hãy chạy training trước.")

In [ ]:
# Export mesh từ checkpoint
!python launch.py \
    --config $save_dir/configs/parsed.yaml \
    --export --gpu 0 \
    resume=$save_dir/ckpts/last.ckpt \
    system.exporter_type=mesh-exporter \
    system.exporter.context_type=cuda \
    system.geometry.isosurface_threshold=15.0

---
## 👁️ 8. Visualize Kết Quả

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Video
import glob
import os

# Hiển thị validation images
val_images = sorted(glob.glob(f"{save_dir}/save/*.png"))
if not val_images:
    val_images = sorted(glob.glob(f"{save_dir}/**/*.png", recursive=True))[-8:]

if val_images:
    n_show = min(8, len(val_images))
    fig, axes = plt.subplots(1, n_show, figsize=(4*n_show, 4))
    if n_show == 1:
        axes = [axes]
    for ax, img_path in zip(axes, val_images[-n_show:]):
        img = plt.imread(img_path)
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(os.path.basename(img_path), fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print("Không tìm thấy hình ảnh validation")

# Hiển thị video 360° nếu có
videos = glob.glob(f"{save_dir}/save/*.mp4")
if videos:
    print(f"\n🎬 Video 360°:")
    display(Video(videos[-1], embed=True, width=512))

---
## 💾 9. Lưu Kết Quả Lên Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Tạo thư mục output trên Drive
drive_output = '/content/drive/MyDrive/TextTo3D_outputs'
os.makedirs(drive_output, exist_ok=True)

# Copy kết quả export
export_dirs = glob.glob(f"{save_dir}/save/it*-export/")
if export_dirs:
    export_dir = export_dirs[-1]
    run_name = os.path.basename(os.path.dirname(save_dir))
    dest = os.path.join(drive_output, run_name)
    !cp -r {export_dir} {dest}
    print(f"✅ Đã lưu mesh đến: {dest}")
else:
    print("❌ Chưa export mesh. Hãy chạy cell Export trước.")

# Copy videos nếu có
for v in glob.glob(f"{save_dir}/save/*.mp4"):
    !cp {v} {drive_output}/
    print(f"✅ Đã lưu video: {os.path.basename(v)}")

---
## 📊 10. Benchmark So Sánh (Optional)

Chạy tất cả các phương pháp với cùng prompt để so sánh.

In [ ]:
import time

benchmark_prompt = "a DSLR photo of a corgi taking a selfi"
max_steps = 5000  # giảm steps để benchmark nhanh

configs = {
    'baseline_sd': 'configs/dreamfusion-sd.yaml',
    'progressive': 'configs/dreamfusion-sd-progressive.yaml',
    'antijanus': 'configs/dreamfusion-sd-antijanus.yaml',
    'optimized': 'configs/optimized-dreamfusion.yaml',
}

results = {}
for name, config in configs.items():
    print(f"\n{'='*60}")
    print(f"  Running: {name} ({config})")
    print(f"{'='*60}")
    start = time.time()
    !python launch.py \
        --config {config} \
        --train --gpu 0 \
        system.prompt_processor.prompt="{benchmark_prompt}" \
        system.prompt_processor.spawn=false \
        trainer.max_steps={max_steps}
    elapsed = time.time() - start
    results[name] = elapsed
    print(f"\n⏱️ {name}: {elapsed/60:.1f} phút")

print("\n" + "="*60)
print("  BENCHMARK RESULTS")
print("="*60)
for name, t in results.items():
    speedup = results.get('baseline_sd', t) / t if t > 0 else 0
    print(f"  {name:20s}: {t/60:6.1f} phút  (x{speedup:.2f})")